# LeetCode 689 — Maximum Sum of 3 Non-Overlapping Subarrays bằng Python

- Link: https://leetcode.com/problems/maximum-sum-of-3-non-overlapping-subarrays/
- Độ khó: **Hard**
- Chủ đề: Tổng tiền tố (prefix sum), Cửa sổ trượt, Mảng "tốt nhất bên trái/bên phải", Quy hoạch động, Truy vết đáp án theo thứ tự từ điển

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxSumOfThreeSubarrays(self, nums: List[int], k: int) -> List[int]:

```

## 1. Giới thiệu & mục tiêu học

Chọn **3 đoạn con liên tiếp, mỗi đoạn dài đúng `k`, không chồng lên nhau**, sao cho tổng lớn nhất. Nghe như phải thử mọi bộ ba vị trí (`O(n³)`), nhưng một mẹo cổ điển — **cố định đoạn ở giữa, tra sẵn đoạn tốt nhất bên trái và bên phải** — đưa bài về `O(n)`.

Cái khó thật sự của bài là chi tiết: đáp án phải là bộ chỉ số **nhỏ nhất theo thứ tự từ điển** khi có nhiều bộ cùng tổng lớn nhất. Chỉ cần đổi `>` thành `>=` ở sai chỗ là sai.

Sau notebook này bạn sẽ:

1. Tính tổng mọi cửa sổ độ dài `k` trong `O(n)` bằng prefix sum / cửa sổ trượt.
2. Viết brute force `O(m³)` và hiểu vì sao duyệt theo thứ tự tăng dần + so sánh `>` cho ra đáp án nhỏ nhất theo từ điển.
3. Xây mảng `left[i]` (cửa sổ tốt nhất trong `[0..i]`) và `right[i]` (tốt nhất trong `[i..m-1]`), xử lý hoà đúng chiều.
4. Viết lời giải một lượt (sliding window ba tầng) với `O(1)` bộ nhớ phụ.
5. Tổng quát hoá thành DP cho `t` đoạn và truy vết đáp án nhỏ nhất theo từ điển.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng số nguyên dương `nums` và số nguyên `k`. Tìm **ba đoạn con không chồng nhau**, mỗi đoạn có độ dài `k`, sao cho **tổng của cả 3 đoạn lớn nhất**. Trả về danh sách **chỉ số bắt đầu** của ba đoạn (đánh số từ 0). Nếu có nhiều đáp án, trả về đáp án **nhỏ nhất theo thứ tự từ điển**.

**Ví dụ 1**

```text
Input:  nums = [1,2,1,2,6,7,5,1], k = 2
Output: [0,3,5]
Giải thích: các đoạn [1,2], [2,6], [7,5] bắt đầu ở 0, 3, 5; tổng 3 + 8 + 12 = 23.
            [2,1] (bắt đầu 1) cũng có tổng 3 nhưng [1,3,5] lớn hơn [0,3,5] theo từ điển.
```

**Ví dụ 2**

```text
Input:  nums = [1,2,1,2,1,2,1,2,1], k = 2
Output: [0,2,4]
```

**Ràng buộc**

- `1 <= nums.length <= 2 * 10^4`
- `1 <= nums[i] < 2^16`
- `1 <= k <= floor(nums.length / 3)`

In [1]:
nums1, k1 = [1, 2, 1, 2, 6, 7, 5, 1], 2
nums2, k2 = [1, 2, 1, 2, 1, 2, 1, 2, 1], 2

def show_choice(nums, k, starts):
    parts = [nums[s:s + k] for s in starts]
    print(f"starts={starts} -> đoạn {parts}, tổng = {sum(map(sum, parts))}")

show_choice(nums1, k1, [0, 3, 5])
show_choice(nums1, k1, [1, 3, 5])
show_choice(nums2, k2, [0, 2, 4])

starts=[0, 3, 5] -> đoạn [[1, 2], [2, 6], [7, 5]], tổng = 23
starts=[1, 3, 5] -> đoạn [[2, 1], [2, 6], [7, 5]], tổng = 23
starts=[0, 2, 4] -> đoạn [[1, 2], [1, 2], [1, 2]], tổng = 9


## 3. Kiến thức nền

### 3.1. Tổng tiền tố (prefix sum)

`pre[i] = nums[0] + … + nums[i-1]` (với `pre[0] = 0`). Khi đó tổng đoạn `nums[a:b]` = `pre[b] - pre[a]`, tính trong `O(1)`.

In [2]:
from itertools import accumulate

nums = [1, 2, 1, 2, 6, 7, 5, 1]
pre = [0] + list(accumulate(nums))
print("pre =", pre)
print("tổng nums[3:5] =", pre[5] - pre[3], "=", sum(nums[3:5]))

pre = [0, 1, 3, 4, 6, 12, 19, 24, 25]
tổng nums[3:5] = 8 = 8


### 3.2. Mảng tổng cửa sổ `W`

`W[i] = sum(nums[i : i + k])` với `i = 0 .. m-1`, `m = n - k + 1` là số cửa sổ. Có thể tính bằng prefix sum, hoặc bằng **cửa sổ trượt**: `W[i] = W[i-1] + nums[i+k-1] - nums[i-1]` (thêm phần tử mới bên phải, bỏ phần tử cũ bên trái).

In [3]:
def window_sums(nums, k):
    W = [sum(nums[:k])]
    for i in range(1, len(nums) - k + 1):
        W.append(W[-1] + nums[i + k - 1] - nums[i - 1])
    return W

W1 = window_sums(nums1, k1)
print("W =", W1)
assert W1 == [pre[i + k1] - pre[i] for i in range(len(nums1) - k1 + 1)]

W = [3, 3, 3, 8, 13, 12, 6]


### 3.3. "Không chồng nhau" theo chỉ số bắt đầu

Ba đoạn bắt đầu ở `a < b < c` không chồng nhau khi và chỉ khi `b >= a + k` và `c >= b + k`. Ví dụ `k = 2`, `a = 0` thì `b` nhỏ nhất là `2`.

### 3.4. So sánh thứ tự từ điển của list

Python so sánh list theo từ điển sẵn: so phần tử đầu, bằng nhau thì so phần tử tiếp.

In [4]:
print([0, 3, 5] < [1, 3, 5], [0, 3, 5] < [0, 4, 5], [0, 3, 5] < [0, 3, 5])
print(min([[1, 3, 5], [0, 3, 6], [0, 3, 5]]))

True True False
[0, 3, 5]


### 3.5. Duyệt theo thứ tự + so sánh `>` giữ lại phần tử **đầu tiên** đạt max

Nếu duyệt các ứng viên theo thứ tự **tăng dần** và chỉ cập nhật khi **lớn hơn hẳn** (`>`), ta giữ lại ứng viên **xuất hiện đầu tiên** trong các ứng viên có cùng giá trị lớn nhất. Dùng `>=` sẽ giữ ứng viên **cuối cùng**.

In [5]:
vals = [3, 7, 2, 7, 5]
best_gt = best_ge = 0
for i, v in enumerate(vals):
    if v > vals[best_gt]:
        best_gt = i
    if v >= vals[best_ge]:
        best_ge = i
print("dùng > :", best_gt, "| dùng >= :", best_ge)

dùng > : 1 | dùng >= : 3


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Đổi bài về mảng `W`

Sau khi có `W`, bài trở thành: **chọn 3 chỉ số `a < b < c` với khoảng cách ≥ `k` sao cho `W[a] + W[b] + W[c]` lớn nhất**.

Với ví dụ 1 (`k = 2`): `W = [3, 3, 3, 8, 13, 12, 6]`.

### 4.2. Tính tay: cố định đoạn giữa

Cố định `b`. Khi đó:

- `a` chọn tự do trong `[0, b - k]` → nên chọn `W[a]` lớn nhất ở đó;
- `c` chọn tự do trong `[b + k, m - 1]` → chọn `W[c]` lớn nhất ở đó.

| `b` | Vùng `a` | max `W[a]` (chỉ số nhỏ nhất) | Vùng `c` | max `W[c]` (chỉ số nhỏ nhất) | Tổng |
|---|---|---|---|---|---|
| 2 | [0..0] | 3 (a=0) | [4..6] | 13 (c=4) | 3 + 3 + 13 = 19 |
| 3 | [0..1] | 3 (a=0) | [5..6] | 12 (c=5) | 3 + 8 + 12 = **23** |
| 4 | [0..2] | 3 (a=0) | [6..6] | 6 (c=6) | 3 + 13 + 6 = 22 |

Max là 23 tại `b = 3` → `[0, 3, 5]`. ✓

Nhận xét then chốt: "max trong `[0, b-k]`" và "max trong `[b+k, m-1]`" có thể **tính sẵn cho mọi vị trí** — đó là mảng `left` và `right`.

In [6]:
W = W1
m = len(W)
for b in range(k1, m - k1):
    a = max(range(0, b - k1 + 1), key=lambda i: (W[i], -i))
    c = max(range(b + k1, m), key=lambda i: (W[i], -i))
    print(f"b={b}: a={a} (W={W[a]}), c={c} (W={W[c]}), tổng={W[a] + W[b] + W[c]}")

b=2: a=0 (W=3), c=4 (W=13), tổng=19
b=3: a=0 (W=3), c=5 (W=12), tổng=23
b=4: a=0 (W=3), c=6 (W=6), tổng=22


Mẹo nhỏ trong code trên: `key=lambda i: (W[i], -i)` — lấy max theo `W[i]`, hoà thì ưu tiên `-i` lớn, tức `i` **nhỏ**.

## 5. Cách 1 — Brute Force: thử mọi bộ ba `(a, b, c)`

**Ý tưởng.** Duyệt `a` tăng dần, `b` từ `a + k`, `c` từ `b + k`. Cập nhật khi tổng **lớn hơn hẳn** → giữ bộ đầu tiên theo thứ tự duyệt = nhỏ nhất theo từ điển (vì ta duyệt đúng theo thứ tự từ điển).

In [7]:
def max3_brute(nums, k):
    W = window_sums(nums, k)
    m = len(W)
    best, ans = -1, None
    for a in range(m):
        for b in range(a + k, m):
            for c in range(b + k, m):
                s = W[a] + W[b] + W[c]
                if s > best:
                    best, ans = s, [a, b, c]
    return ans

print(max3_brute(nums1, k1))   # [0, 3, 5]
print(max3_brute(nums2, k2))   # [0, 2, 4]

[0, 3, 5]
[0, 2, 4]


**Độ phức tạp.** `O(m³)` với `m ≈ n`; `n = 2·10^4` → khoảng `10^12 / 6` phép — không thể. Bộ nhớ `O(n)`.

**Nút thắt.** Với mỗi `(a, b)` ta lại quét toàn bộ `c` để tìm max — nhưng "max của `W` trên một đoạn hậu tố" có thể tính sẵn một lần. Tương tự cho `a`.

## 6. Bước đột phá

**Insight:** cố định **đoạn giữa** `b`. Hai đoạn còn lại độc lập nhau: đoạn trái chỉ phụ thuộc "tiền tố `[0..b-k]`", đoạn phải chỉ phụ thuộc "hậu tố `[b+k..m-1]`".

- `left[i]` = chỉ số của `W` lớn nhất trong `W[0..i]`; **hoà thì chỉ số nhỏ nhất** → duyệt từ trái, cập nhật khi `W[i] > W[left[i-1]]` (dấu `>`).
- `right[i]` = chỉ số của `W` lớn nhất trong `W[i..m-1]`; **hoà thì chỉ số nhỏ nhất** → duyệt từ **phải**, cập nhật khi `W[i] >= W[right[i+1]]` (dấu `>=`, vì đi từ phải sang, chỉ số nhỏ hơn đến sau và phải "thắng" khi hoà).

Quy tắc: với mỗi `b` từ `k` đến `m - k - 1`: `a = left[b - k]`, `c = right[b + k]`; **nếu** tổng lớn hơn hẳn best **thì** cập nhật.

**Vì sao vẫn đảm bảo nhỏ nhất theo từ điển?** (chứng minh đầy đủ ở mục 10) — ý chính: với `b` cố định, `left`/`right` đã chọn `a`, `c` nhỏ nhất; và nếu một bộ tối ưu khác có `a` nhỏ hơn thì `W[a]` của nó phải nhỏ hơn hẳn, ghép `a` tốt hơn với đoạn giữa/phải của nó sẽ vượt max — mâu thuẫn.

**Insight 2 (một lượt):** khi trượt ba cửa sổ đồng thời (`i`, `i+k`, `i+2k`), ta có thể duy trì "tốt nhất cho 1 đoạn", "tốt nhất cho 2 đoạn", "tốt nhất cho 3 đoạn" cùng lúc — không cần mảng.

**Insight 3 (tổng quát):** "tốt nhất cho `t` đoạn trong hậu tố bắt đầu từ `i`" là một DP; truy vết tham lam từ trái cho đáp án nhỏ nhất theo từ điển.

## 7. Các cách tiếp theo

Tổng cộng 4 cách:

| Cách | Ý tưởng | Time | Space |
|---|---|---|---|
| 1 | Brute force mọi bộ ba | `O(n³)` | `O(n)` |
| 2 | `W` + mảng `left`/`right` + duyệt đoạn giữa | `O(n)` | `O(n)` |
| 3 | Một lượt, trượt ba cửa sổ, giữ best 1/2/3 đoạn | `O(n)` | `O(1)` phụ |
| 4 | DP tổng quát `t` đoạn + truy vết | `O(t·n)` | `O(t·n)` |

### 7.1. Cách 2 — Mảng `left` / `right`

In [8]:
def max3_left_right(nums, k):
    W = window_sums(nums, k)
    m = len(W)

    left = [0] * m
    for i in range(1, m):
        left[i] = i if W[i] > W[left[i - 1]] else left[i - 1]

    right = [m - 1] * m
    for i in range(m - 2, -1, -1):
        right[i] = i if W[i] >= W[right[i + 1]] else right[i + 1]

    best, ans = -1, None
    for b in range(k, m - k):
        a, c = left[b - k], right[b + k]
        s = W[a] + W[b] + W[c]
        if s > best:
            best, ans = s, [a, b, c]
    return ans

print(max3_left_right(nums1, k1))
print(max3_left_right(nums2, k2))

[0, 3, 5]
[0, 2, 4]


Giải thích từng phần:

- `left[0] = 0` (khởi tạo `[0] * m`); `left[i]` hoặc là `i` (nếu `W[i]` lớn hơn hẳn max cũ) hoặc giữ nguyên `left[i-1]`.
- `right[m-1] = m-1`; đi ngược, `>=` để khi hoà thì chỉ số **nhỏ hơn** (đến sau) thắng.
- `range(k, m - k)`: đoạn giữa phải chừa đủ chỗ: `b - k >= 0` và `b + k <= m - 1`.
- So sánh `s > best` với `b` tăng dần → hoà thì giữ `b` nhỏ nhất.

**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def maxSumOfThreeSubarrays(self, nums: List[int], k: int) -> List[int]:
        W = [sum(nums[:k])]
        for i in range(1, len(nums) - k + 1):
            W.append(W[-1] + nums[i + k - 1] - nums[i - 1])
        m = len(W)
        left = [0] * m
        for i in range(1, m):
            left[i] = i if W[i] > W[left[i - 1]] else left[i - 1]
        right = [m - 1] * m
        for i in range(m - 2, -1, -1):
            right[i] = i if W[i] >= W[right[i + 1]] else right[i + 1]
        best, ans = -1, None
        for b in range(k, m - k):
            a, c = left[b - k], right[b + k]
            s = W[a] + W[b] + W[c]
            if s > best:
                best, ans = s, [a, b, c]
        return ans
```

**Độ phức tạp.** Ba vòng `O(m)` → `O(n)` thời gian; ba mảng độ dài `m` → `O(n)` bộ nhớ.

### 7.2. Cách 3 — Một lượt, trượt ba cửa sổ

**Giải thích.** Ba cửa sổ trượt song song, bắt đầu ở `i`, `i + k`, `i + 2k` với tổng `w1`, `w2`, `w3`. Duy trì:

- `best1`, `idx1`: cửa sổ tốt nhất (1 đoạn) trong số các cửa sổ bắt đầu `≤ i`;
- `best2`, `idx2`: cặp tốt nhất (2 đoạn), đoạn thứ hai bắt đầu `≤ i + k`;
- `best3`, `idx3`: bộ ba tốt nhất, đoạn thứ ba bắt đầu `≤ i + 2k`.

Mỗi bước: cập nhật `best1` bằng `w1`; rồi `best2` bằng `best1 + w2` (đoạn thứ hai là cửa sổ `i + k`, ghép với đoạn tốt nhất bắt đầu `≤ i`); rồi `best3` bằng `best2 + w3`. Luôn dùng `>` để giữ ứng viên sớm nhất.

In [9]:
def max3_one_pass(nums, k):
    w1 = sum(nums[0:k])
    w2 = sum(nums[k:2 * k])
    w3 = sum(nums[2 * k:3 * k])
    best1, idx1 = w1, 0
    best2, idx2 = w1 + w2, (0, k)
    best3, idx3 = w1 + w2 + w3, (0, k, 2 * k)
    for i in range(1, len(nums) - 3 * k + 1):
        w1 += nums[i + k - 1] - nums[i - 1]
        w2 += nums[i + 2 * k - 1] - nums[i + k - 1]
        w3 += nums[i + 3 * k - 1] - nums[i + 2 * k - 1]
        if w1 > best1:
            best1, idx1 = w1, i
        if best1 + w2 > best2:
            best2, idx2 = best1 + w2, (idx1, i + k)
        if best2 + w3 > best3:
            best3, idx3 = best2 + w3, idx2 + (i + 2 * k,)
    return list(idx3)

print(max3_one_pass(nums1, k1))
print(max3_one_pass(nums2, k2))

[0, 3, 5]
[0, 2, 4]


Chú ý thứ tự cập nhật trong một vòng: `best1` trước, rồi `best2` dùng `best1` **mới** (được phép vì cửa sổ `i` và cửa sổ `i + k` không chồng nhau), rồi `best3` dùng `best2` mới (cửa sổ `i + k` và `i + 2k` không chồng nhau).

**Code nộp LeetCode (Cách 3):**

```python
class Solution:
    def maxSumOfThreeSubarrays(self, nums: List[int], k: int) -> List[int]:
        w1, w2, w3 = sum(nums[:k]), sum(nums[k:2*k]), sum(nums[2*k:3*k])
        best1, idx1 = w1, 0
        best2, idx2 = w1 + w2, (0, k)
        best3, idx3 = w1 + w2 + w3, (0, k, 2 * k)
        for i in range(1, len(nums) - 3 * k + 1):
            w1 += nums[i + k - 1] - nums[i - 1]
            w2 += nums[i + 2*k - 1] - nums[i + k - 1]
            w3 += nums[i + 3*k - 1] - nums[i + 2*k - 1]
            if w1 > best1:
                best1, idx1 = w1, i
            if best1 + w2 > best2:
                best2, idx2 = best1 + w2, (idx1, i + k)
            if best2 + w3 > best3:
                best3, idx3 = best2 + w3, idx2 + (i + 2*k,)
        return list(idx3)
```

**Độ phức tạp.** `O(n)` thời gian, `O(1)` bộ nhớ phụ.

### 7.3. Cách 4 — DP tổng quát cho `t` đoạn + truy vết

**Giải thích.** Định nghĩa `f[t][i]` = tổng lớn nhất khi chọn `t` cửa sổ không chồng nhau, **mọi chỉ số bắt đầu ≥ `i`** (`f[0][i] = 0`; `f[t][i] = -∞` nếu không đủ chỗ).

> `f[t][i] = max( f[t][i+1],  W[i] + f[t-1][i+k] )`  (bỏ cửa sổ `i`, hoặc chọn nó)

Đáp án tổng = `f[3][0]`. **Truy vết nhỏ nhất theo từ điển:** đi từ trái, với `t` đoạn còn lại và vị trí `pos`, chọn `i ≥ pos` **nhỏ nhất** sao cho `W[i] + f[t-1][i+k] == f[t][pos]` — chọn được `i` nhỏ nhất ở mỗi bước chính là định nghĩa thứ tự từ điển.

In [10]:
def max_t_subarrays(nums, k, t):
    W = window_sums(nums, k)
    m, n = len(W), len(nums)
    NEG = float('-inf')
    f = [[0] * (n + 2)] + [[NEG] * (n + 2) for _ in range(t)]
    for j in range(1, t + 1):
        for i in range(m - 1, -1, -1):
            take = W[i] + f[j - 1][i + k]
            f[j][i] = max(f[j][i + 1], take)
    # truy vết
    ans, pos = [], 0
    for j in range(t, 0, -1):
        i = pos
        while W[i] + f[j - 1][i + k] != f[j][pos]:
            i += 1
        ans.append(i)
        pos = i + k
    return ans

def max3_dp(nums, k):
    return max_t_subarrays(nums, k, 3)

print(max3_dp(nums1, k1), max3_dp(nums2, k2))
print("2 đoạn:", max_t_subarrays(nums1, k1, 2), "| 1 đoạn:", max_t_subarrays(nums1, k1, 1))

[0, 3, 5] [0, 2, 4]
2 đoạn: [3, 5] | 1 đoạn: [4]


Giải thích:

- Mảng `f[j]` có độ dài `n + 2` để `f[j-1][i + k]` (với `i + k` có thể bằng `n`) và `f[j][i + 1]` không vượt chỉ số; các ô ngoài vùng hợp lệ là `-∞` (với `j ≥ 1`) hoặc `0` (với `j = 0`).
- Vòng `while` truy vết: luôn dừng vì tồn tại ít nhất một `i` đạt giá trị `f[j][pos]` (theo định nghĩa max).
- Vì sao điều kiện truy vết đúng: `f[j][pos]` là max trên mọi `i ≥ pos` của `W[i] + f[j-1][i+k]` (mở rộng công thức truy hồi) — nên chọn `i` nhỏ nhất đạt max là hợp lệ.

**Code nộp LeetCode (Cách 4):**

```python
class Solution:
    def maxSumOfThreeSubarrays(self, nums: List[int], k: int) -> List[int]:
        n = len(nums)
        W = [sum(nums[:k])]
        for i in range(1, n - k + 1):
            W.append(W[-1] + nums[i + k - 1] - nums[i - 1])
        m, T, NEG = len(W), 3, float('-inf')
        f = [[0] * (n + 2)] + [[NEG] * (n + 2) for _ in range(T)]
        for j in range(1, T + 1):
            for i in range(m - 1, -1, -1):
                f[j][i] = max(f[j][i + 1], W[i] + f[j - 1][i + k])
        ans, pos = [], 0
        for j in range(T, 0, -1):
            i = pos
            while W[i] + f[j - 1][i + k] != f[j][pos]:
                i += 1
            ans.append(i)
            pos = i + k
        return ans
```

**Độ phức tạp.** `O(t · n)` thời gian và bộ nhớ (`t = 3` → `O(n)`). Lợi thế: đổi `3` thành `t` bất kỳ.

## 8. TRACE từng bước

### 8.1. Bảng `W`, `left`, `right` cho ví dụ 1

`nums = [1,2,1,2,6,7,5,1]`, `k = 2`:

| i | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|---|
| `W[i]` | 3 | 3 | 3 | 8 | 13 | 12 | 6 |
| `left[i]` | 0 | 0 | 0 | 3 | 4 | 4 | 4 |
| `right[i]` | 4 | 4 | 4 | 4 | 4 | 5 | 6 |

Chú ý `left[1] = left[2] = 0` (hoà, giữ chỉ số nhỏ) và `right[0..3] = 4`.

### 8.2. `trace_left_right`

In [11]:
def trace_left_right(nums, k):
    W = window_sums(nums, k)
    m = len(W)
    left = [0] * m
    for i in range(1, m):
        left[i] = i if W[i] > W[left[i - 1]] else left[i - 1]
    right = [m - 1] * m
    for i in range(m - 2, -1, -1):
        right[i] = i if W[i] >= W[right[i + 1]] else right[i + 1]
    print("i     :", list(range(m)))
    print("W     :", W)
    print("left  :", left)
    print("right :", right)
    best, ans = -1, None
    for b in range(k, m - k):
        a, c = left[b - k], right[b + k]
        s = W[a] + W[b] + W[c]
        mark = ""
        if s > best:
            best, ans = s, [a, b, c]
            mark = "  <- cập nhật"
        print(f"  b={b}: a=left[{b - k}]={a}, c=right[{b + k}]={c}, tổng={W[a]}+{W[b]}+{W[c]}={s}{mark}")
    print("đáp án:", ans, "tổng", best)
    return ans

trace_left_right(nums1, k1)
print()
trace_left_right(nums2, k2)

i     : [0, 1, 2, 3, 4, 5, 6]
W     : [3, 3, 3, 8, 13, 12, 6]
left  : [0, 0, 0, 3, 4, 4, 4]
right : [4, 4, 4, 4, 4, 5, 6]
  b=2: a=left[0]=0, c=right[4]=4, tổng=3+3+13=19  <- cập nhật
  b=3: a=left[1]=0, c=right[5]=5, tổng=3+8+12=23  <- cập nhật
  b=4: a=left[2]=0, c=right[6]=6, tổng=3+13+6=22
đáp án: [0, 3, 5] tổng 23

i     : [0, 1, 2, 3, 4, 5, 6, 7]
W     : [3, 3, 3, 3, 3, 3, 3, 3]
left  : [0, 0, 0, 0, 0, 0, 0, 0]
right : [0, 1, 2, 3, 4, 5, 6, 7]
  b=2: a=left[0]=0, c=right[4]=4, tổng=3+3+3=9  <- cập nhật
  b=3: a=left[1]=0, c=right[5]=5, tổng=3+3+3=9
  b=4: a=left[2]=0, c=right[6]=6, tổng=3+3+3=9
  b=5: a=left[3]=0, c=right[7]=7, tổng=3+3+3=9
đáp án: [0, 2, 4] tổng 9


[0, 2, 4]

### 8.3. `trace_one_pass` — ba cửa sổ trượt

In [12]:
def trace_one_pass(nums, k):
    w1, w2, w3 = sum(nums[:k]), sum(nums[k:2 * k]), sum(nums[2 * k:3 * k])
    best1, idx1 = w1, 0
    best2, idx2 = w1 + w2, (0, k)
    best3, idx3 = w1 + w2 + w3, (0, k, 2 * k)
    print(f"i=0: w=({w1},{w2},{w3}) best1={best1}@{idx1} best2={best2}@{idx2} best3={best3}@{idx3}")
    for i in range(1, len(nums) - 3 * k + 1):
        w1 += nums[i + k - 1] - nums[i - 1]
        w2 += nums[i + 2 * k - 1] - nums[i + k - 1]
        w3 += nums[i + 3 * k - 1] - nums[i + 2 * k - 1]
        if w1 > best1:
            best1, idx1 = w1, i
        if best1 + w2 > best2:
            best2, idx2 = best1 + w2, (idx1, i + k)
        if best2 + w3 > best3:
            best3, idx3 = best2 + w3, idx2 + (i + 2 * k,)
        print(f"i={i}: w=({w1},{w2},{w3}) best1={best1}@{idx1} best2={best2}@{idx2} best3={best3}@{idx3}")
    return list(idx3)

trace_one_pass(nums1, k1)

i=0: w=(3,3,13) best1=3@0 best2=6@(0, 2) best3=19@(0, 2, 4)
i=1: w=(3,8,12) best1=3@0 best2=11@(0, 3) best3=23@(0, 3, 5)
i=2: w=(3,13,6) best1=3@0 best2=16@(0, 4) best3=23@(0, 3, 5)


[0, 3, 5]

### 8.4. Trace DP: bảng `f[t][i]` và truy vết

In [13]:
def trace_dp(nums, k, t=3):
    W = window_sums(nums, k)
    m, n = len(W), len(nums)
    NEG = float('-inf')
    f = [[0] * (n + 2)] + [[NEG] * (n + 2) for _ in range(t)]
    for j in range(1, t + 1):
        for i in range(m - 1, -1, -1):
            f[j][i] = max(f[j][i + 1], W[i] + f[j - 1][i + k])
    show = lambda v: ' -∞' if v == NEG else f"{v:>3}"
    print("i      :", ' '.join(f"{i:>3}" for i in range(m)))
    for j in range(1, t + 1):
        print(f"f[{j}][i] :", ' '.join(show(f[j][i]) for i in range(m)))
    ans, pos = [], 0
    for j in range(t, 0, -1):
        i = pos
        while W[i] + f[j - 1][i + k] != f[j][pos]:
            i += 1
        print(f"  còn {j} đoạn, pos={pos}: cần {f[j][pos]}; i nhỏ nhất với W[i]+f[{j-1}][i+k] = {W[i]}+{f[j-1][i+k]} -> i={i}")
        ans.append(i)
        pos = i + k
    print("đáp án:", ans)
    return ans

trace_dp(nums1, k1)

i      :   0   1   2   3   4   5   6
f[1][i] :  13  13  13  13  13  12   6
f[2][i] :  20  20  20  20  19  -∞  -∞
f[3][i] :  23  23  22  -∞  -∞  -∞  -∞
  còn 3 đoạn, pos=0: cần 23; i nhỏ nhất với W[i]+f[2][i+k] = 3+20 -> i=0
  còn 2 đoạn, pos=2: cần 20; i nhỏ nhất với W[i]+f[1][i+k] = 8+12 -> i=3
  còn 1 đoạn, pos=5: cần 12; i nhỏ nhất với W[i]+f[0][i+k] = 12+0 -> i=5
đáp án: [0, 3, 5]


[0, 3, 5]

## 9. Code nộp LeetCode

Chọn **Cách 2** (mảng `left`/`right`) — dễ giải thích, dễ kiểm tra hoà. Cách 3 cũng tốt nếu muốn `O(1)` bộ nhớ.

In [14]:
from typing import List

class Solution:
    def maxSumOfThreeSubarrays(self, nums: List[int], k: int) -> List[int]:
        W = [sum(nums[:k])]
        for i in range(1, len(nums) - k + 1):
            W.append(W[-1] + nums[i + k - 1] - nums[i - 1])
        m = len(W)

        left = [0] * m
        for i in range(1, m):
            left[i] = i if W[i] > W[left[i - 1]] else left[i - 1]

        right = [m - 1] * m
        for i in range(m - 2, -1, -1):
            right[i] = i if W[i] >= W[right[i + 1]] else right[i + 1]

        best, ans = -1, None
        for b in range(k, m - k):
            a, c = left[b - k], right[b + k]
            s = W[a] + W[b] + W[c]
            if s > best:
                best, ans = s, [a, b, c]
        return ans

assert Solution().maxSumOfThreeSubarrays([1, 2, 1, 2, 6, 7, 5, 1], 2) == [0, 3, 5]
assert Solution().maxSumOfThreeSubarrays([1, 2, 1, 2, 1, 2, 1, 2, 1], 2) == [0, 2, 4]
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Tổng lớn nhất.** Mọi bộ hợp lệ `(a, b, c)` có `a ≤ b - k` và `c ≥ b + k`. Với `b` cố định, `W[a] ≤ W[left[b-k]]` và `W[c] ≤ W[right[b+k]]`. Vậy tổng của mọi bộ ≤ tổng thuật toán xét tại `b` → max trên mọi `b` là tổng lớn nhất.

**Nhỏ nhất theo từ điển.** Gọi `(a*, b*, c*)` là đáp án thuật toán (với `b*` là `b` **nhỏ nhất** đạt max, nhờ dấu `>`), và giả sử có bộ tối ưu `(a, b, c)` nhỏ hơn theo từ điển.

Trước hết một nhận xét: với **mọi** bộ tối ưu `(a, b, c)`, thuật toán tại chính `b` đã tính được tổng ≥ tổng bộ đó = max, nên `b ≥ b*`.

1. Nếu `a < a*`: vì `a < a* ≤ b* - k`, chỉ số `a` nằm trong vùng `[0..b*-k]`, mà `a* = left[b* - k]` là chỉ số **nhỏ nhất** đạt max trên vùng đó, nên `W[a] < W[a*]` (nếu bằng thì `left` đã chọn `a`). Xét bộ `(a*, b, c)`: hợp lệ vì `a* ≤ b* - k ≤ b - k`. Tổng của nó `W[a*] + W[b] + W[c] > W[a] + W[b] + W[c] = max` — mâu thuẫn.
2. Nếu `a = a*` và `b < b*`: bộ `(a, b, c)` đạt max với đoạn giữa `b` nhỏ hơn `b*`, nhưng thuật toán tại `b` đã tính được tổng ≥ tổng bộ này = max, nên `b*` không thể là `b` nhỏ nhất đạt max — mâu thuẫn.
3. Nếu `a = a*`, `b = b*`, `c < c*`: `c* = right[b* + k]` là chỉ số nhỏ nhất đạt max trên hậu tố, và `W[c] = W[c*]` (vì tổng bằng nhau) → `c* ≤ c` — mâu thuẫn.

**Cách 3** tương đương Cách 2 nhưng gộp ba mảng vào ba biến chạy; **Cách 4** đúng theo công thức truy hồi và truy vết chọn chỉ số nhỏ nhất ở mỗi bước. Cả hai được đối chiếu với brute force ở mục 12.

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(n³)` | `O(n)` |
| 2. left/right | `O(n)` | `O(n)` |
| 3. Một lượt | `O(n)` | `O(1)` phụ |
| 4. DP `t` đoạn | `O(t·n)` | `O(t·n)` |

Đo trên `n = 20000`:

In [15]:
import random, time

random.seed(689)
big = [random.randint(1, 65535) for _ in range(20000)]
for k in [1, 50, 6666]:
    res = []
    for name, fn in [("left/right", max3_left_right), ("one pass", max3_one_pass), ("dp", max3_dp)]:
        t0 = time.perf_counter()
        res.append(fn(big, k))
        print(f"k={k:>5} {name:<10}: {res[-1]}  {time.perf_counter() - t0:.3f}s")
    assert res[0] == res[1] == res[2]

k=    1 left/right: [2588, 9443, 16013]  0.007s
k=    1 one pass  : [2588, 9443, 16013]  0.008s
k=    1 dp        : [2588, 9443, 16013]  0.011s
k=   50 left/right: [3075, 10220, 10504]  0.005s
k=   50 one pass  : [3075, 10220, 10504]  0.006s
k=   50 dp        : [3075, 10220, 10504]  0.007s
k= 6666 left/right: [2, 6668, 13334]  0.003s
k= 6666 one pass  : [2, 6668, 13334]  0.000s
k= 6666 dp        : [2, 6668, 13334]  0.004s


## 12. Bộ test

1. Ví dụ đề.
2. Edge case: `n = 3k` (chỉ một cách chọn), mọi phần tử bằng nhau (hoà toàn bộ → `[0, k, 2k]`), `k = 1`, max nằm ở cuối mảng.
3. Ngẫu nhiên có seed với **giá trị nhỏ (1..3)** để tạo nhiều hoà — kiểm tra kỹ thứ tự từ điển; đối chiếu brute force.

In [16]:
METHODS = [max3_left_right, max3_one_pass, max3_dp, lambda a, k: Solution().maxSumOfThreeSubarrays(a, k)]

def check(nums, k, expected=None):
    ref = max3_brute(nums, k)
    for f in METHODS:
        got = f(nums, k)
        assert got == ref, (nums, k, got, ref)
    if expected is not None:
        assert ref == expected, (nums, k, ref, expected)
    return ref

check([1, 2, 1, 2, 6, 7, 5, 1], 2, [0, 3, 5])
check([1, 2, 1, 2, 1, 2, 1, 2, 1], 2, [0, 2, 4])
check([4, 5, 6], 1, [0, 1, 2])                    # n = 3k
check([7] * 10, 3, [0, 3, 6])                     # mọi phần tử bằng nhau
check([1, 1, 1, 1, 9, 9, 9], 1, [4, 5, 6])        # max ở cuối
check([9, 1, 9, 1, 9, 1, 9], 1, [0, 2, 4])        # hoà: chọn sớm nhất
check([3, 3, 3, 1, 1, 1, 3, 3, 3], 3, [0, 3, 6])
print("Ví dụ + edge case: OK")

random.seed(2026)
for _ in range(3000):
    k = random.randint(1, 3)
    n = random.randint(3 * k, 3 * k + 9)
    nums = [random.randint(1, 3) for _ in range(n)]
    check(nums, k)
print("3000 bộ ngẫu nhiên (nhiều hoà): mọi cách khớp brute force")

Ví dụ + edge case: OK
3000 bộ ngẫu nhiên (nhiều hoà): mọi cách khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1: Mảng `right` dùng `>` (hoà thì giữ chỉ số lớn)

In [17]:
def max3_wrong_right(nums, k):
    W = window_sums(nums, k)
    m = len(W)
    left = [0] * m
    for i in range(1, m):
        left[i] = i if W[i] > W[left[i - 1]] else left[i - 1]
    right = [m - 1] * m
    for i in range(m - 2, -1, -1):
        right[i] = i if W[i] > W[right[i + 1]] else right[i + 1]     # SAI: phải là >=
    best, ans = -1, None
    for b in range(k, m - k):
        a, c = left[b - k], right[b + k]
        if W[a] + W[b] + W[c] > best:
            best, ans = W[a] + W[b] + W[c], [a, b, c]
    return ans

x = [7] * 10
print("sai:", max3_wrong_right(x, 3), "| đúng:", max3_brute(x, 3))

sai: [0, 3, 7] | đúng: [0, 3, 6]


**Vì sao sai:** đi từ phải sang, phần tử đến sau có chỉ số **nhỏ hơn**; muốn hoà thì chỉ số nhỏ thắng nên phải dùng `>=`.

### Lỗi 2: Mảng `left` dùng `>=` (hoà thì giữ chỉ số lớn)

In [18]:
def max3_wrong_left(nums, k):
    W = window_sums(nums, k)
    m = len(W)
    left = [0] * m
    for i in range(1, m):
        left[i] = i if W[i] >= W[left[i - 1]] else left[i - 1]       # SAI: phải là >
    right = [m - 1] * m
    for i in range(m - 2, -1, -1):
        right[i] = i if W[i] >= W[right[i + 1]] else right[i + 1]
    best, ans = -1, None
    for b in range(k, m - k):
        a, c = left[b - k], right[b + k]
        if W[a] + W[b] + W[c] > best:
            best, ans = W[a] + W[b] + W[c], [a, b, c]
    return ans

print("sai:", max3_wrong_left(nums1, k1), "| đúng:", max3_brute(nums1, k1))

sai: [1, 3, 5] | đúng: [0, 3, 5]


**Vì sao sai:** đúng như ví dụ 1 trong đề: `[1, 3, 5]` có cùng tổng nhưng lớn hơn `[0, 3, 5]` theo từ điển. Từ trái sang, `>` giữ chỉ số nhỏ.

### Lỗi 3: Lấy `left[b - 1]` / `right[b + 1]` — các đoạn chồng nhau

In [19]:
def max3_overlap(nums, k):
    W = window_sums(nums, k)
    m = len(W)
    left = [0] * m
    for i in range(1, m):
        left[i] = i if W[i] > W[left[i - 1]] else left[i - 1]
    right = [m - 1] * m
    for i in range(m - 2, -1, -1):
        right[i] = i if W[i] >= W[right[i + 1]] else right[i + 1]
    best, ans = -1, None
    for b in range(1, m - 1):
        a, c = left[b - 1], right[b + 1]                               # SAI: phải là b-k, b+k
        if W[a] + W[b] + W[c] > best:
            best, ans = W[a] + W[b] + W[c], [a, b, c]
    return ans

print("sai:", max3_overlap(nums1, k1), "| đúng:", max3_brute(nums1, k1))

sai: [3, 4, 5] | đúng: [0, 3, 5]


**Vì sao sai:** đoạn bắt đầu ở `b - 1` chồng lên đoạn bắt đầu ở `b` khi `k ≥ 2`. Hai đoạn không chồng nhau cần chỉ số bắt đầu cách nhau **ít nhất `k`**.

### Lỗi 4: Tính tổng cửa sổ bằng `sum(nums[i:i+k])` trong vòng lặp

Đúng nhưng `O(n·k)`; với `n = 2·10^4`, `k = 6666` là ~`10^8` phép → rất chậm. Luôn dùng prefix sum hoặc cửa sổ trượt.

## 14. Mẹo & nhận dạng pattern

- **"Chọn vài đoạn không chồng nhau, tối đa tổng"** → tính `W` trước, rồi "cố định đoạn giữa + tốt nhất trái/phải" (với 3 đoạn) hoặc DP `f[t][i]` (với `t` đoạn).
- **"Tốt nhất trong tiền tố / hậu tố"** → mảng `left`/`right` (prefix max / suffix max), rất hay gặp: *Trapping Rain Water*, *Best Time to Buy and Sell Stock III*.
- **"Nhiều đáp án → nhỏ nhất theo từ điển"** → để ý chiều duyệt và dấu `>` vs `>=`; truy vết DP **từ trái** và chọn chỉ số nhỏ nhất.
- **Ba cửa sổ trượt song song** là mẹo gọn gàng để bỏ mảng phụ.

## 15. Biến thể

1. **`t` đoạn bất kỳ** → Cách 4 (`max_t_subarrays`).
2. **Hai đoạn với độ dài khác nhau `L` và `M`** — LeetCode 1031 *Maximum Sum of Two Non-Overlapping Subarrays* (đoạn `L` có thể trước hoặc sau đoạn `M`).
3. **Chỉ cần tổng, không cần chỉ số** → bỏ phần truy vết.
4. **Đáp án lớn nhất theo từ điển** → đảo dấu so sánh (hoặc truy vết chọn chỉ số lớn nhất).
5. **Phần tử có thể âm** → vẫn đúng với Cách 2/3/4 (khởi tạo `best` bằng `-∞` thay vì `-1`).

Ví dụ biến thể 2 (LeetCode 1031):

In [20]:
def max_two_no_overlap(nums, L, M):
    pre = [0] + list(accumulate(nums))
    def best(first, second):
        res = 0
        best_first = 0
        for end in range(first + second, len(nums) + 1):
            best_first = max(best_first, pre[end - second] - pre[end - second - first])
            res = max(res, best_first + pre[end] - pre[end - second])
        return res
    return max(best(L, M), best(M, L))

def max_two_brute(nums, L, M):
    n, res = len(nums), 0
    for i in range(n - L + 1):
        for j in range(n - M + 1):
            if i + L <= j or j + M <= i:
                res = max(res, sum(nums[i:i + L]) + sum(nums[j:j + M]))
    return res

print(max_two_no_overlap([0, 6, 5, 2, 2, 5, 1, 9, 4], 1, 2))   # 20
random.seed(1031)
for _ in range(300):
    L, M = random.randint(1, 3), random.randint(1, 3)
    arr = [random.randint(0, 9) for _ in range(random.randint(L + M, L + M + 6))]
    assert max_two_no_overlap(arr, L, M) == max_two_brute(arr, L, M)
print("biến thể 1031 khớp brute force")

20
biến thể 1031 khớp brute force


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | Mọi bộ `(a, b, c)` theo thứ tự từ điển | `O(n³)` | `O(n)` | Trọng tài khi test |
| 2. left/right | Cố định đoạn giữa, tra max tiền tố/hậu tố | `O(n)` | `O(n)` | Lời giải chuẩn, dễ giải thích hoà |
| 3. Một lượt | Ba cửa sổ trượt, best 1/2/3 đoạn | `O(n)` | `O(1)` | Muốn tiết kiệm bộ nhớ, code gọn |
| 4. DP `t` đoạn | `f[t][i]` + truy vết từ trái | `O(t·n)` | `O(t·n)` | Khi số đoạn thay đổi / tổng quát |

## 17. Bài tập

**Bài 1 (đoán output).** `maxSumOfThreeSubarrays([1,1,1,1,1,1], 2)`, `([5,1,1,5,1,1,5], 1)`, `([1,2,3,4,5,6,7,8,9], 3)`?

**Bài 2 (tính tay).** Lập bảng `W`, `left`, `right` cho `nums = [4,3,2,1,9,8,1,1]`, `k = 2` và tìm đáp án.

**Bài 3 (điền TODO).** Hoàn thiện `build_right(W)` (chú ý dấu so sánh!).

**Bài 4 (tự viết lại).** Viết lại `my_max3(nums, k)` theo Cách 2.

**Bài 5 (mini challenge).** Viết `max_t_sum(nums, k, t)` chỉ trả về **tổng** lớn nhất của `t` đoạn không chồng nhau, dùng `O(n)` bộ nhớ (chỉ giữ hai hàng DP).

In [21]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def build_right(W):
    # TODO: right[m-1] = m-1; đi ngược i = m-2..0; right[i] = i nếu W[i] >= W[right[i+1]]
    pass

# Bài 4 — TODO
def my_max3(nums, k):
    # TODO: W, left, right; duyệt b in range(k, m-k); cập nhật khi tổng > best
    pass

# Bài 5 — TODO
def max_t_sum(nums, k, t):
    # TODO: prev = [0]*(n+2); lặp t lần: cur = [-inf]*(n+2); cur[i] = max(cur[i+1], W[i] + prev[i+k])
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: mọi cửa sổ có tổng bằng nhau thì sao?
- Mức 2: `[5,1,1,5,1,1,5], k=1`: ba số 5 ở 0, 3, 6.
- Mức 3: `[0,2,4]`, `[0,3,6]`, `[0,3,6]` (k=3 và n=9 chỉ có một cách).

**Bài 2.**
- Mức 1: `W = [7, 5, 3, 10, 17, 9, 2]`.
- Mức 2: `left = [0,0,0,3,4,4,4]`, `right = [4,4,4,4,4,5,6]`.
- Mức 3: `b=2`: 7+3+17 = 27; `b=3`: 7+10+9 = 26; `b=4`: 7+17+2 = 26 → `[0, 2, 4]`.

**Bài 3.**
- Mức 1: khởi tạo `[m-1] * m`.
- Mức 2: duyệt `range(m - 2, -1, -1)`.
- Mức 3: dùng `>=`.

**Bài 4.**
- Mức 1: tính `W` bằng cửa sổ trượt.
- Mức 2: `left` dùng `>`, `right` dùng `>=`.
- Mức 3: `a = left[b-k]`, `c = right[b+k]`.

**Bài 5.**
- Mức 1: `f[0][*] = 0`.
- Mức 2: chỉ cần hàng `j - 1` để tính hàng `j`.
- Mức 3: đáp án `prev[0]` sau `t` vòng.

## 19. Đáp án tham khảo

In [22]:
S = Solution()
# Bài 1
assert S.maxSumOfThreeSubarrays([1, 1, 1, 1, 1, 1], 2) == [0, 2, 4]
assert S.maxSumOfThreeSubarrays([5, 1, 1, 5, 1, 1, 5], 1) == [0, 3, 6]
assert S.maxSumOfThreeSubarrays([1, 2, 3, 4, 5, 6, 7, 8, 9], 3) == [0, 3, 6]

# Bài 2
assert window_sums([4, 3, 2, 1, 9, 8, 1, 1], 2) == [7, 5, 3, 10, 17, 9, 2]
assert S.maxSumOfThreeSubarrays([4, 3, 2, 1, 9, 8, 1, 1], 2) == [0, 2, 4]

# Bài 3
def build_right_ans(W):
    m = len(W)
    right = [m - 1] * m
    for i in range(m - 2, -1, -1):
        right[i] = i if W[i] >= W[right[i + 1]] else right[i + 1]
    return right

assert build_right_ans([3, 3, 3, 8, 13, 12, 6]) == [4, 4, 4, 4, 4, 5, 6]
assert build_right_ans([5, 5, 5]) == [0, 1, 2]
print("Bài 1–3 OK")

Bài 1–3 OK


In [23]:
# Bài 4
def my_max3_ans(nums, k):
    W = window_sums(nums, k)
    m = len(W)
    left = [0] * m
    for i in range(1, m):
        left[i] = i if W[i] > W[left[i - 1]] else left[i - 1]
    right = build_right_ans(W)
    best, ans = -1, None
    for b in range(k, m - k):
        s = W[left[b - k]] + W[b] + W[right[b + k]]
        if s > best:
            best, ans = s, [left[b - k], b, right[b + k]]
    return ans

# Bài 5
def max_t_sum_ans(nums, k, t):
    n = len(nums)
    W = window_sums(nums, k)
    prev = [0] * (n + 2)
    for _ in range(t):
        cur = [float('-inf')] * (n + 2)
        for i in range(len(W) - 1, -1, -1):
            cur[i] = max(cur[i + 1], W[i] + prev[i + k])
        prev = cur
    return prev[0]

random.seed(99)
for _ in range(1500):
    k = random.randint(1, 3)
    nums = [random.randint(1, 4) for _ in range(random.randint(3 * k, 3 * k + 8))]
    ref = max3_brute(nums, k)
    assert my_max3_ans(nums, k) == ref
    assert max_t_sum_ans(nums, k, 3) == sum(sum(nums[s:s + k]) for s in ref)
print("Bài 4–5 OK (đối chiếu brute force)")

Bài 4–5 OK (đối chiếu brute force)


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "3 đoạn không chồng nhau":**

```python
W = tổng mọi cửa sổ độ dài k (cửa sổ trượt)
left[i]  = argmax W[0..i]     # từ trái, dùng  >  (hoà giữ chỉ số nhỏ)
right[i] = argmax W[i..m-1]   # từ phải, dùng >= (hoà giữ chỉ số nhỏ)
for b in range(k, m - k):
    a, c = left[b-k], right[b+k]
    nếu W[a]+W[b]+W[c] > best: cập nhật
```

**Template DP `t` đoạn:** `f[j][i] = max(f[j][i+1], W[i] + f[j-1][i+k])`; truy vết từ trái chọn `i` nhỏ nhất đạt `f[j][pos]`.

**Checklist trước khi submit:**

- [ ] Tính `W` trong `O(n)` (không `sum` lặp lại).
- [ ] `left` dùng `>`, `right` dùng `>=` (cả hai đều ưu tiên chỉ số nhỏ khi hoà).
- [ ] Đoạn giữa duyệt `range(k, m - k)`; tra `left[b-k]`, `right[b+k]`.
- [ ] Cập nhật best bằng `>` (giữ `b` nhỏ nhất).
- [ ] Test hoà: `[1,2,1,2,1,2,1,2,1], k=2 → [0,2,4]`; mọi phần tử bằng nhau → `[0,k,2k]`.
- [ ] Trả về **list** chỉ số bắt đầu (không phải tổng).
- [ ] Tên method đúng: `maxSumOfThreeSubarrays(self, nums, k)`.